# 00b_cost_matrix: what each kind of mistake costs, in money

**Owner:** Liza

**Covers:** Step 4 in the README (brief Part A.3): the cost of approving a loan that defaults vs declining a good borrower, in shillings and cedis, the "decline above X%" threshold that follows, and how it moves under other assumptions. Write-up: `learnings/04_cost_matrix.md`.

**Team rules for this notebook**

1. **Explain everything as plainly as possible.** The rest of the group will not redo this part; they will rely on what you write here. Every step gets a markdown cell before it: what we are doing, why, and what the result means in plain English. No unexplained jargon.
2. **Test every reasonable alternative.** Do not run one option and move on. For each step, try the alternatives, put the results side by side in a table, and state which one wins and why.
3. **It must run top to bottom** (Kernel > Restart & Run All) from a fresh clone with `data/` in place.

In [1]:
import pandas as pd
import numpy as np

DATA = '../data/zindi/'

## Step 1: What does a typical loan look like?

**What we are doing:** finding the usual loan size and length, in Kenya and in Ghana.

**Why:** the costs of a mistake depend on how much money is lent and for how long. We need a realistic "typical loan" to put numbers on.

**Where the numbers come from:**
- **Kenya:** `Train.csv` (all Kenyan). Amounts are assumed to be in **Kenyan shillings (KES)**.
- **Ghana:** the Ghanaian rows of `Test.csv`. Amounts are assumed to be in **Ghana cedis (GHS)**. (The files don't state the currency; see `docs/01_data.md`.)

**Average or middle value?** We use the **median**: the middle loan, where half are smaller and half are bigger. The **mean** (the ordinary average) is pulled up by a few huge loans. For example, one 23-million loan raises the average of thousands of 5,000 loans a lot, but doesn't change the middle one.

In [2]:
train = pd.read_csv(DATA + 'Train.csv')
test = pd.read_csv(DATA + 'Test.csv')
kenya = train
ghana = test[test.country_id == 'Ghana']

def describe(loans):
    return pd.Series({
        'loans': len(loans),
        'middle amount (median)': loans.Total_Amount.median(),
        'average amount (mean)': loans.Total_Amount.mean(),
        'middle length in days (median)': loans.duration.median(),
        'share of loans lasting 7 days or less': (loans.duration <= 7).mean(),
    })

typical = pd.DataFrame({'Kenya (KES)': describe(kenya), 'Ghana (GHS)': describe(ghana)})
typical.loc['share of loans lasting 7 days or less'] = typical.loc['share of loans lasting 7 days or less'] * 100  # as %
typical.round(0)

,Kenya (KES),Ghana (GHS)
loans,68654.0,3525.0
middle amount (median),5249.0,3002.0
average amount (mean),14837.0,15537.0
middle length in days (median),7.0,7.0
share of loans lasting 7 days or less,95.0,86.0


**What this means:**
- **Kenya:** the middle loan is about **5,250 shillings** (about $38 at 139.8 KES per USD), and 95% of loans last a week or less.
- **Ghana:** the middle loan is about **3,000 cedis** (about $270 at 11.0 GHS per USD), and 86% last a week or less.
- **The averages are about 3–5 times the middle values,** because a few very big loans pull them up. So the median is the fairer "typical loan".
- If the currency assumption is right, a typical Ghanaian loan is about **7 times bigger in dollars** than a Kenyan one. That's surprising for small mobile-money loans, so we flag it.

### How spread out are the loan sizes?

The middle loan hides a wide range. This shows the amount at several points: e.g. "10%" is the amount that 10% of loans are smaller than.

In [3]:
points = [0.10, 0.25, 0.50, 0.75, 0.90]
spread = pd.DataFrame({
    'Kenya (KES)': kenya.Total_Amount.quantile(points),
    'Ghana (GHS)': ghana.Total_Amount.quantile(points),
})
spread.index = [f'{p:.0%} of loans are smaller than' for p in points]
spread.round(0)

,Kenya (KES),Ghana (GHS)
10% of loans are smaller than,1069.0,1193.0
25% of loans are smaller than,2295.0,1581.0
50% of loans are smaller than,5249.0,3002.0
75% of loans are smaller than,11450.0,5924.0
90% of loans are smaller than,25051.0,55321.0


**What this means:** loan sizes are very spread out. In Kenya, 1 in 10 loans is under about 1,070 shillings and 1 in 10 is over 25,000. In Ghana, the biggest 10% are over 55,000 cedis. A mistake on a big loan costs far more than on a small one, so later we work in **percentages of the loan** as well as in money.

### Does loan size depend on loan length?

Longer loans might be bigger. If so, one "typical loan" won't fit every product.

In [4]:
bands = pd.cut(kenya.duration, [0, 7, 14, 30, 90, 2000],
               labels=['1–7 days', '8–14 days', '15–30 days', '31–90 days', 'over 90 days'])
by_length = kenya.groupby(bands, observed=True).agg(loans=('Total_Amount', 'size'), middle_amount_KES=('Total_Amount', 'median'))
by_length['share of loans'] = (by_length.loans / len(kenya)).map('{:.1%}'.format)
by_length

,loans,middle_amount_KES,share of loans
duration,,,
1–7 days,65041,5000.0,94.7%
8–14 days,1572,17800.0,2.3%
15–30 days,1147,18000.0,1.7%
31–90 days,742,80000.0,1.1%
over 90 days,152,258717.2,0.2%


**What this means:** yes, longer loans are much bigger. The middle 7-day loan is 5,000 shillings, but the middle 31–90-day loan is 80,000. Still, 95% of loans are 7-day loans.

**Decision for Step 1: our typical loan is a 7-day loan of 5,000 KES in Kenya and 3,000 GHS in Ghana** (rounded medians). Because the costs we work out next grow with the loan size, the "decline above X%" rule will mostly depend on percentages (fee % and loss %), not on the exact amount. Step 4 checks this.

## Step 2: How much does NkwaPay earn when a loan is repaid?

**What we are doing:** measuring the **fee**: the extra money a borrower pays back on top of what they borrowed.

**fee % = (amount to repay − amount borrowed) ÷ amount borrowed.** A loan of 5,000 with 5,035 to repay has a fee of 35 shillings, or 0.7%.

**Why only repaid loans?** The feature audit (`learnings/03_feature_audit.md`) found that `Total_Amount_to_Repay` includes penalties added *after* someone fails to pay. For repaid loans there are no penalties, so the column shows the real fee agreed on day 1.

**Why it matters:** the fee is what NkwaPay loses when it turns away a good borrower. It's one of the two numbers in the cost matrix.

In [5]:
repaid = kenya[kenya.target == 0].copy()
repaid['fee_pct'] = (repaid.Total_Amount_to_Repay - repaid.Total_Amount) / repaid.Total_Amount * 100
repaid['fee_money'] = repaid.Total_Amount_to_Repay - repaid.Total_Amount

print(f'Repaid Kenyan loans: {len(repaid):,}')
print(f'Middle fee (median):  {repaid.fee_pct.median():.2f}%   = {repaid.fee_money.median():,.0f} KES')
print(f'Average fee (mean):   {repaid.fee_pct.mean():.2f}%   = {repaid.fee_money.mean():,.0f} KES')
print(f'Loans with NO fee at all: {(repaid.fee_pct == 0).mean():.0%}')

Repaid Kenyan loans: 67,396
Middle fee (median):  0.76%   = 32 KES
Average fee (mean):   1.68%   = 504 KES
Loans with NO fee at all: 36%


**What this means:** the middle repaid loan pays a fee of only **0.76% (32 shillings)**. The average is higher (1.68%) because longer, bigger loans pay bigger fees. And **more than 1 in 3 repaid loans pay no fee at all.**

### Does the fee depend on loan length?

Longer loans should cost more, the same way a longer phone contract costs more in total.

In [6]:
repaid_bands = pd.cut(repaid.duration, [0, 7, 14, 30, 90, 2000],
                      labels=['1–7 days', '8–14 days', '15–30 days', '31–90 days', 'over 90 days'])
fee_by_length = repaid.groupby(repaid_bands, observed=True).agg(
    loans=('fee_pct', 'size'),
    middle_fee_pct=('fee_pct', 'median'),
    share_with_no_fee=('fee_pct', lambda f: (f == 0).mean() * 100),
    middle_amount_KES=('Total_Amount', 'median'),
)
fee_by_length['middle_fee_KES'] = fee_by_length.middle_fee_pct / 100 * fee_by_length.middle_amount_KES
fee_by_length.round(1)

,loans,middle_fee_pct,share_with_no_fee,middle_amount_KES,middle_fee_KES
duration,,,,,
1–7 days,64250,0.7,37.4,5000.0,36.0
8–14 days,1443,5.5,0.7,17420.0,962.7
15–30 days,995,7.0,1.3,20000.0,1400.0
31–90 days,624,11.2,4.0,74384.0,8368.2
over 90 days,84,15.0,0.0,290000.0,43500.0


**What this means:**
- Longer loans pay much higher fees: 0.7% for a week, up to 15% for loans over 90 days.
- The "no fee" loans are almost all **7-day loans**: 37% of them pay nothing, against 0–4% for longer loans.

### Fees on 7-day loans, for loans that do charge a fee

Many loans charge nothing. Here we look only at 7-day loans that charge *something*, to see what a "normal" fee is.

In [7]:
week = repaid[repaid.duration == 7]
charged = week[week.fee_pct > 0]
print(f'7-day repaid loans: {len(week):,}; with a fee: {len(charged):,} ({len(charged) / len(week):.0%})')
print('Fee % among those that charge one:')
print(charged.fee_pct.quantile([0.25, 0.5, 0.75]).round(2).rename(lambda q: f'  {q:.0%} of loans pay less than').to_string())

7-day repaid loans: 64,182; with a fee: 40,104 (62%)
Fee % among those that charge one:
25% of loans pay less than    0.80
50% of loans pay less than    2.01
75% of loans pay less than    3.07


**What this means:** among 7-day loans that *do* charge a fee, the middle fee is **2.0%** (about 100 shillings on 5,000).

**The "no fee" puzzle:** a lender doesn't usually lend for free. The likely explanations are that the fee was taken a different way (e.g. deducted from the money sent, or charged by the mobile-money platform), or that these were promotional loans. The data can't tell us. So the 0.7% middle fee probably **understates** what NkwaPay really earns, and 2.0% is a more realistic fee for a loan that charges one.

**Side note for the fairness and regulation work:** 2% for one week, repeated every week, is about 104% a year. That's worth comparing with the Bank of Ghana Directive's rules on pricing later.

### Ghana's fees

Ghana has no labels, so we can't separate repaid loans from defaults. But only about 2% of loans default, so the **middle** fee is still a fair guide (a few penalised loans can't move the middle much).

In [8]:
ghana_fee_pct = (ghana.Total_Amount_to_Repay - ghana.Total_Amount) / ghana.Total_Amount * 100
print(f'Ghana middle fee (median): {ghana_fee_pct.median():.2f}%   = {(ghana.Total_Amount_to_Repay - ghana.Total_Amount).median():,.0f} GHS')
print(f'Ghana loans with NO fee: {(ghana_fee_pct == 0).mean():.0%}')
print('Most common Ghana fees:')
print(ghana_fee_pct.round(1).value_counts(normalize=True).head(5).map('{:.0%}'.format).rename_axis('fee %').to_string())

Ghana middle fee (median): 3.00%   = 87 GHS
Ghana loans with NO fee: 1%
Most common Ghana fees:
fee %
3.0     86%
30.0     2%
33.0     2%
27.0     1%
54.0     1%


**What this means:** Ghana's pricing is very simple: **86% of loans charge exactly 3%**, and almost none are free. On a typical 3,000-cedi loan that's **90 cedis**. (The few loans showing 27–54% are probably penalised defaults, or longer loans.)

### The fee options we'll carry forward

| Option | Kenya (7-day loan of 5,000 KES) | Ghana (7-day loan of 3,000 GHS) | When it's right |
|---|---|---|---|
| **Low:** middle fee of all loans | 0.7% = **36 KES** | — | If free loans really earn nothing |
| **Main:** middle fee of loans that charge one | 2.0% = **100 KES** | 3.0% = **90 GHS** | If free loans earn their fee some other way |

**Decision for Step 2:** use **2.0% (100 KES)** for Kenya and **3.0% (90 GHS)** for Ghana as the main fee, and test the low 0.7% in the sensitivity check (Step 5).

## Step 3: What does each mistake cost?

There are four possible outcomes for every loan application:

| | Borrower would **repay** | Borrower would **default** |
|---|---|---|
| NkwaPay **approves** | ✅ Good: NkwaPay earns the fee | ❌ **False negative (FN):** NkwaPay loses the money lent |
| NkwaPay **declines** | ❌ **False positive (FP):** NkwaPay loses the fee, and the borrower goes without credit | ✅ Good: the loss is avoided |

**How to remember the names:** the model's job is to *flag risky borrowers*. "Positive" = flagged as risky, "negative" = not flagged.
- **False negative (FN)** = the model *didn't* flag someone, but it *should* have. **NkwaPay approved someone who won't pay back.**
- **False positive (FP)** = the model flagged someone, but it *shouldn't* have. **NkwaPay turned away someone who would have paid back.**

From here on, every time FN or FP comes up, a short reminder in brackets says what it means.

**The two costs:**
- **Cost of FN** (approving someone who won't pay back) **= money lent − money recovered later + cost of chasing the debt**
- **Cost of FP** (turning away someone who would pay back) **= the fee NkwaPay would have earned**

### What the data can and can't tell us

| Number | From the data? | How we get it |
|---|---|---|
| Money lent | ✅ yes | Step 1: 5,000 KES / 3,000 GHS |
| Fee | ✅ yes | Step 2: 2.0% / 3.0% |
| Money recovered after a default | ❌ no | **Assumption**, tested in Step 5 |
| Cost of chasing the debt | ❌ no | **Assumption**, tested in Step 5 |

The data has no column saying how much was recovered from defaulters, or what collection cost. So we must assume those two numbers, say why, and show how much the answer changes if we're wrong.

### Our assumptions

**Recovery: 10% of the loan.** These loans have no collateral (nothing to take back, like a house or a car) and are tiny, so chasing them hard isn't worth much. We assume NkwaPay gets back only about 1 in 10 shillings lent. Step 5 also tries 0% (nothing recovered) and 30%.

**Collection cost: about $1.50 per defaulted loan** (≈ 200 KES at 139.8 KES/USD; ≈ 16 GHS at 11.0 GHS/USD). This covers a few weeks of SMS reminders, a few phone calls from staff, and an agent follow-up. We use the same dollar cost in both countries, because the work involved is similar. Step 5 also tries double this.

These are our own estimates, not facts from the data. They should be replaced if NkwaPay (or a published source) gives real figures.

In [9]:
FN_LABEL = "cost of FN (approve someone who won't pay back)"
FP_LABEL = "cost of FP (turn away someone who would pay back)"

costs = pd.DataFrame({
    'Kenya (KES)': {'money lent': 5000, 'fee %': 2.0, 'recovered %': 10, 'collection cost': 200},
    'Ghana (GHS)': {'money lent': 3000, 'fee %': 3.0, 'recovered %': 10, 'collection cost': 16},
})

def cost_of_mistakes(c):
    fee = c['money lent'] * c['fee %'] / 100
    recovered = c['money lent'] * c['recovered %'] / 100
    fn = c['money lent'] - recovered + c['collection cost']
    fp = fee
    return pd.Series({'fee earned if repaid': fee, 'recovered after default': recovered,
                      FN_LABEL: fn, FP_LABEL: fp,
                      'times bigger: cost of FN ÷ cost of FP': fn / fp})

cost_matrix = costs.apply(cost_of_mistakes)
cost_matrix.round(1)

,Kenya (KES),Ghana (GHS)
fee earned if repaid,100.0,90.0
recovered after default,500.0,300.0
cost of FN (approve someone who won't pay back),4700.0,2716.0
cost of FP (turn away someone who would pay back),100.0,90.0
times bigger: cost of FN ÷ cost of FP,47.0,30.2


**What this means:**
- **Kenya:** approving a borrower who defaults costs **4,700 shillings** (5,000 lent − 500 recovered + 200 to chase it). Declining a good borrower costs **100 shillings** (the lost fee). So one bad approval costs as much as turning away **47** good borrowers.
- **Ghana:** approving a defaulter costs **2,716 cedis**; declining a good borrower costs **90 cedis**. One bad approval costs as much as turning away about **30** good borrowers.
- Ghana's ratio is smaller because its fee is higher (3% vs 2%): each good loan earns more, so turning one away hurts more.

### The repeat-customer question

The cost of FP (turning away someone who would pay back) above counts only **one** lost fee. But NkwaPay's customers come back again and again. If NkwaPay turns away a good customer, they may go to another lender and never come back, so NkwaPay loses **all their future fees**.

How many more loans does a customer usually take *after* any given loan? We count, for each loan, how many later loans the same customer took.

In [10]:
by_customer = kenya.sort_values('disbursement_date').groupby('customer_id')
later_loans = by_customer.cumcount(ascending=False)  # loans this customer took after this one
print(f'Loans per customer: middle {kenya.groupby("customer_id").size().median():.0f}, average {kenya.groupby("customer_id").size().mean():.1f}')
print(f'Later loans after a typical loan: middle {later_loans.median():.0f}, average {later_loans.mean():.1f}')

Loans per customer: middle 6, average 10.5
Later loans after a typical loan: middle 8, average 12.8


**What this means:** after a typical loan, the same customer takes about **8 more loans** (the middle value). That number counts only loans inside our data window, so the real figure is probably higher.

(Why is this bigger than the 6 loans per customer? Frequent borrowers have many loans, so a "typical loan" more often belongs to a frequent borrower.)

**An alternative cost of FP (turning away someone who would pay back), counting lost repeat business:** suppose that half of declined good customers leave for good. Then declining one costs the fee now, plus half of 8 future fees:

- **Kenya:** 100 + 0.5 × 8 × 100 = **500 KES**
- **Ghana:** 90 + 0.5 × 8 × 90 = **450 GHS** (assuming Ghanaian customers return as often as Kenyan ones)

The "half leave" is an assumption. This is a fuller picture of what turning someone away costs, so Step 5 tests it alongside the simple version.

**The cost the money doesn't show:** a declined borrower also goes without credit they may badly need, for school fees, stock for a small shop, or an emergency. The brief asks us to remember this. It isn't in NkwaPay's cost, but it is part of why being too strict is harmful, and it comes back in the fairness work.

**Decision for Step 3:**

| | Kenya | Ghana |
|---|---|---|
| **Cost of FN** (approve someone who won't pay back) | **4,700 KES** | **2,716 GHS** |
| **Cost of FP** (turn away someone who would pay back), simple | **100 KES** | **90 GHS** |
| Cost of FP (turn away someone who would pay back), with lost repeat business | 500 KES | 450 GHS |

## Step 4: From costs to a rule: "decline if the chance of default is above X%"

**What we are doing:** turning the two costs from Step 3 into one cut-off number.

**A reminder of the two costs (Kenya):**

| | What happened | Cost |
|---|---|---|
| **FN** (false negative) | NkwaPay **approved someone who won't pay back** | **4,700 KES** (5,000 lent − 500 recovered + 200 to chase it) |
| **FP** (false positive) | NkwaPay **turned away someone who would have paid back** | **100 KES** (the lost fee) |

**The idea:** the model gives each applicant a **chance of default**, called *p*. For example, *p* = 3% means "out of 100 people like this, about 3 won't pay back".

### The easy way to see the cut-off: "1 in 48"

One person who doesn't pay back loses 4,700 KES. Each person who pays back earns 100 KES. So **one bad loan wipes out the fees from 47 good loans** (4,700 ÷ 100 = 47).

Picture 48 similar applicants:
- If **exactly 1 of the 48** won't pay back, the 47 good loans earn 47 × 100 = 4,700 and the 1 bad loan loses 4,700. **NkwaPay breaks even.**
- If **fewer than 1 in 48** won't pay back, NkwaPay makes money → **approve**.
- If **more than 1 in 48** won't pay back, NkwaPay loses money → **decline**.

**1 in 48 = 2.1%. That is the cut-off.**

### The same thing as a formula

For one applicant with chance *p* of not paying back, NkwaPay compares the expected (average) cost of each choice:

| Choice | What could go wrong | Expected cost |
|---|---|---|
| **Approve** | they don't pay back (chance *p*), costing the FN amount | *p* × 4,700 |
| **Decline** | they would have paid back (chance 1 − *p*), costing the FP amount | (1 − *p*) × 100 |

The cut-off is the *p* where both choices cost the same:

```
p × 4,700        = (1 − p) × 100
p × 4,700        = 100 − p × 100
p × 4,700 + p × 100 = 100
p × 4,800        = 100
p                = 100 ÷ 4,800 = 0.021 = 2.1%
```

**Check at exactly 2.1%:** approving costs 0.021 × 4,700 ≈ **98**, and declining costs 0.979 × 100 ≈ **98**. They're equal. Above 2.1%, approving costs more, so decline.

In general:

> **decline when *p* is above: cost of FP ÷ (cost of FP + cost of FN)**
> = (what turning away a good borrower costs) ÷ (that + what approving a bad borrower costs)

This is the formula from the brief's Appendix A. The code below works it out for both countries.

In [11]:
def threshold(fn, fp):
    # the cut-off: cost of FP ÷ (cost of FP + cost of FN)
    return fp / (fp + fn)

rule = pd.DataFrame({
    country: {
        FN_LABEL: cost_matrix.loc[FN_LABEL, country],
        FP_LABEL: cost_matrix.loc[FP_LABEL, country],
    } for country in cost_matrix.columns
}).T
rule['decline above (%)'] = threshold(rule[FN_LABEL], rule[FP_LABEL]) * 100
rule.round(2)

,cost of FN (approve someone who won't pay back),cost of FP (turn away someone who would pay back),decline above (%)
Kenya (KES),4700.0,100.0,2.08
Ghana (GHS),2716.0,90.0,3.21


**What this means:**
- **Kenya: decline if the chance of default is above about 2.1%.**
- **Ghana: decline if the chance of default is above about 3.2%.**

Ghana's cut-off is higher because its fee is higher, so each good loan is worth more and NkwaPay can afford a bit more risk.

### A worked example (Kenya)

Two applicants, using the Kenyan costs: FN (approving someone who won't pay back) = 4,700 KES, and FP (turning away someone who would pay back) = 100 KES.

In [12]:
for p in [0.01, 0.05]:
    approve = p * 4700
    decline = (1 - p) * 100
    choice = 'APPROVE' if approve < decline else 'DECLINE'
    print(f'Chance of default {p:.0%}:  expected cost if approved = {approve:6.1f} KES,  if declined = {decline:6.1f} KES  →  {choice}')

Chance of default 1%:  expected cost if approved =   47.0 KES,  if declined =   99.0 KES  →  APPROVE
Chance of default 5%:  expected cost if approved =  235.0 KES,  if declined =   95.0 KES  →  DECLINE


**What this means:** at a 1% chance of default, approving is cheaper (47 KES expected loss vs 99 KES of lost fees), so approve. At 5%, declining is cheaper (95 vs 235), so decline. The cut-off of 2.1% is where the two expected costs are equal.

### Does the cut-off depend on the loan size?

The fee and the money lost both grow with the loan, but the collection cost (200 KES) is the same for every loan. So for small loans, chasing a default costs relatively more. Here is the Kenyan cut-off for different loan sizes, with everything else the same.

In [13]:
sizes = [1000, 2500, 5000, 11500, 25000]
by_size = pd.DataFrame({'loan size (KES)': sizes})
by_size[FN_LABEL] = by_size['loan size (KES)'] * 0.90 + 200   # lent − 10% recovered + 200 to chase
by_size[FP_LABEL] = by_size['loan size (KES)'] * 0.02          # the 2% fee
by_size['decline above (%)'] = threshold(by_size[FN_LABEL], by_size[FP_LABEL]) * 100
by_size.round(2)

,loan size (KES),cost of FN (approve someone who won't pay back),cost of FP (turn away someone who would pay back),decline above (%)
0,1000,1100.0,20.0,1.79
1,2500,2450.0,50.0,2.00
2,5000,4700.0,100.0,2.08
3,11500,10550.0,230.0,2.13
4,25000,22700.0,500.0,2.16


**What this means:** the cut-off barely moves with loan size: from 1.8% for a 1,000-KES loan to 2.2% for a 25,000-KES loan. So **one cut-off per country is good enough** for typical 7-day loans. (Small loans get a slightly stricter cut-off, because chasing a default costs the same 200 KES whatever the size.)

### How strict is this rule in practice?

We don't have the model yet (that's Germain's step 7). As a rough illustration, suppose the model only knew the loan length. Its chance of default would be the default share for that length (from the feature audit). Which loans would the rule approve?

In [14]:
rates = kenya.groupby(bands, observed=True).target.agg(loans='size', default_share='mean')
rates['default share (%)'] = rates.default_share * 100
cutoff = rule.loc['Kenya (KES)', 'decline above (%)']
rates['decision'] = np.where(rates['default share (%)'] > cutoff, 'DECLINE', 'approve')
print(f'Kenyan cut-off: decline above {cutoff:.2f}%')
rates[['loans', 'default share (%)', 'decision']].round(2)

Kenyan cut-off: decline above 2.08%


,loans,default share (%),decision
duration,,,
1–7 days,65041,1.22,approve
8–14 days,1572,8.21,DECLINE
15–30 days,1147,13.25,DECLINE
31–90 days,742,15.90,DECLINE
over 90 days,152,44.74,DECLINE


**But longer loans charge bigger fees** (Step 2: 5.5% to 15%), so they deserve their own cut-off. Here each loan length uses its own middle fee and middle amount:

In [15]:
fee_rates = fee_by_length.middle_fee_pct / 100
fee_rates['1–7 days'] = 0.02  # our main 7-day fee from Step 2 (the 0.7% median includes the free loans)
amounts = fee_by_length.middle_amount_KES
own = pd.DataFrame({
    'middle fee (%)': fee_rates * 100,
    'own cut-off (%)': threshold(amounts * 0.90 + 200, amounts * fee_rates) * 100,
    'default share (%)': rates['default share (%)'],
})
own['decision'] = np.where(own['default share (%)'] > own['own cut-off (%)'], 'DECLINE', 'approve')
own.round(2)

,middle fee (%),own cut-off (%),default share (%),decision
duration,,,,
1–7 days,2.00,2.08,1.22,approve
8–14 days,5.53,5.72,8.21,DECLINE
15–30 days,7.00,7.14,13.25,DECLINE
31–90 days,11.25,11.08,15.90,DECLINE
over 90 days,15.00,14.28,44.74,DECLINE


**What this means:**
- With its own fee, each longer loan type gets a higher cut-off (about 6% to 14%), because it earns more.
- But their default shares (8% to 45%) are still above their cut-offs. So in this rough illustration, a model that only knew the loan length would approve the 7-day loans and decline the longer ones.
- The real model knows more than the length, so it will separate risky and safe borrowers *within* each length. This table only shows the rule isn't crazy: it approves 95% of loans.

### ⚠️ Important for Germain (step 7)

The rule only works if the model's chances are **calibrated**: when it says 2%, about 2 in 100 such people really default.
- A model trained with `class_weight="balanced"` gives **inflated** chances, because it pretends defaulters are as common as repayers. Its "50%" doesn't mean 50%.
- So the probabilities must be **calibrated** (Platt or isotonic, as the brief asks) **before** applying the 2.1% / 3.2% cut-off.
- If calibration isn't possible, pick the cut-off on the model's scores that gives the **lowest total cost** on validation data, using the cost of FN (approving someone who won't pay back) and the cost of FP (turning away someone who would pay back) from Step 3.

**Decision for Step 4:**

| | Kenya | Ghana |
|---|---|---|
| **Decline if the chance of default is above** | **2.1%** | **3.2%** |
| Cost of FN (approve someone who won't pay back) | 4,700 KES | 2,716 GHS |
| Cost of FP (turn away someone who would pay back) | 100 KES | 90 GHS |
| The "1 in N" view | 1 in 48 | 1 in 31 |

**What "1 in N" means:** NkwaPay breaks even when 1 borrower in every N doesn't pay back. Fewer bad borrowers than that → approving makes money; more → declining is cheaper.
- **Kenya, 1 in 48:** one bad loan loses 4,700 KES, which wipes out the 100-KES fee from 47 good loans. So in a group of 48, 1 bad loan and 47 good ones break even. 1 ÷ 48 = 2.1%.
- **Ghana, 1 in 31:** one bad loan loses 2,716 GHS, which wipes out the 90-GHS fee from about 30 good loans (2,716 ÷ 90 = 30.2). So about 1 bad loan in every 31 breaks even. 1 ÷ 31 = 3.2%.

For loans longer than 7 days, a cut-off based on their own fee is more accurate (table above).

## Step 5: What if our guesses are wrong?

**What we are doing:** changing one assumption at a time and seeing how far the cut-off moves. The brief asks for at least two alternatives (Appendix A, "Sensitivity check").

**Why:** recovery and collection cost are our own guesses, the fee has two possible readings (Step 2), and the cost of FP (turning away someone who would pay back) could include lost repeat business (Step 3). If the cut-off barely moves, our answer is safe. If it moves a lot, we must say which assumption matters most.

**The scenarios:**

| Scenario | What changes |
|---|---|
| Main | Our chosen numbers (Steps 1–3) |
| Nothing recovered | 0% recovered instead of 10% |
| More recovered | 30% recovered instead of 10% |
| Chasing costs double | collection cost × 2 |
| Low fee (Kenya only) | fee 0.7% instead of 2.0% (free loans really earn nothing) |
| Lost repeat business | cost of FP (turning away someone who would pay back) also includes their future fees (half of declined customers leave, 8 future loans) |
| Strictest | low fee + nothing recovered |
| Most lenient | lost repeat business + 30% recovered |

**Reading the table:** each row shows the cost of FN (approving someone who won't pay back), the cost of FP (turning away someone who would pay back), and the cut-off that follows: cost of FP ÷ (cost of FP + cost of FN).

**The "rough check" column** uses the illustration from Step 4: 7-day loans default about 1.22% of the time. If the cut-off falls *below* that, even ordinary 7-day loans would be declined, and NkwaPay would turn away almost everyone.

In [16]:
def scenario(lent, fee_pct, recovered_pct, collection, repeat_business=False):
    fee = lent * fee_pct / 100
    fn = lent * (1 - recovered_pct / 100) + collection
    fp = fee + (0.5 * 8 * fee if repeat_business else 0)
    return fn, fp, threshold(fn, fp) * 100

base = {'Kenya': dict(lent=5000, fee_pct=2.0, recovered_pct=10, collection=200),
        'Ghana': dict(lent=3000, fee_pct=3.0, recovered_pct=10, collection=16)}
changes = {
    'Main': {},
    'Nothing recovered': dict(recovered_pct=0),
    'More recovered (30%)': dict(recovered_pct=30),
    'Chasing costs double': 'double',
    'Low fee (0.7%)': dict(fee_pct=0.7),
    'Lost repeat business': dict(repeat_business=True),
    'Strictest (low fee + nothing recovered)': dict(fee_pct=0.7, recovered_pct=0),
    'Most lenient (repeat business + 30% recovered)': dict(repeat_business=True, recovered_pct=30),
}
rows = []
for name, change in changes.items():
    for country, b in base.items():
        if name.startswith(('Low fee', 'Strictest')) and country == 'Ghana':
            continue  # Ghana's fee is a clear 3%; the free-loan puzzle is Kenyan only
        settings = {**b, 'collection': b['collection'] * 2} if change == 'double' else {**b, **change}
        fn, fp, cut = scenario(**settings)
        rows.append({'scenario': name, 'country': country, FN_LABEL: fn, FP_LABEL: fp,
                     'decline above (%)': cut,
                     'rough check: 7-day loans (1.22% default)': 'approved' if cut > 1.22 else 'DECLINED'})
sensitivity = pd.DataFrame(rows).set_index(['scenario', 'country'])
sensitivity.round(2)

cost of FN (approve someone who won't pay back)  \
scenario                                       country                                                    
Main                                           Kenya                                             4700.0   
                                               Ghana                                             2716.0   
Nothing recovered                              Kenya                                             5200.0   
                                               Ghana                                             3016.0   
More recovered (30%)                           Kenya                                             3700.0   
                                               Ghana                                             2116.0   
Chasing costs double                           Kenya                                             4900.0   
                                               Ghana                                             2732.0   
Low fee (0.7%)                                 Kenya                                             4700.0   
Lost repeat business                           Kenya                                             4700.0   
                                               Ghana                                             2716.0   
Strictest (low fee + nothing recovered)        Kenya                                             5200.0   
Most lenient (repeat business + 30% recovered) Kenya                                             3700.0   
                                               Ghana                                             2116.0   

                                                        cost of FP (turn away someone who would pay back)  \
scenario                                       country                                                      
Main                                           Kenya                                                100.0   
                                               Ghana                                                 90.0   
Nothing recovered                              Kenya                                                100.0   
                                               Ghana                                                 90.0   
More recovered (30%)                           Kenya                                                100.0   
                                               Ghana                                                 90.0   
Chasing costs double                           Kenya                                                100.0   
                                               Ghana                                                 90.0   
Low fee (0.7%)                                 Kenya                                                 35.0   
Lost repeat business                           Kenya                                                500.0   
                                               Ghana                                                450.0   
Strictest (low fee + nothing recovered)        Kenya                                                 35.0   
Most lenient (repeat business + 30% recovered) Kenya                                                500.0   
                                               Ghana                                                450.0   

                                                        decline above (%)  \
scenario                                       country                      
Main                                           Kenya                 2.08   
                                               Ghana                 3.21   
Nothing recovered                              Kenya                 1.89   
                                               Ghana                 2.90   
More recovered (30%)                           Kenya                 2.63   
                                               Ghana       

**What this means:**

1. **Our two guesses about defaults barely matter.** Changing recovery (0% to 30%) or doubling the chasing cost moves Kenya's cut-off only between **1.9% and 2.6%**, and Ghana's between **2.9% and 4.1%**. Every one of these still approves ordinary 7-day loans. So the answer is **safe against those guesses**.
2. **The fee matters a lot.** If the free Kenyan loans really earn nothing (fee 0.7%), the cut-off falls to **0.7%**, below the 1.22% default share of ordinary 7-day loans. NkwaPay would then decline almost everyone. That's not a realistic way to run a lender, which supports our choice of 2.0% as the main fee.
3. **Counting lost repeat business matters most.** If turning a good customer away also loses their future loans, the cut-off rises to about **10% (Kenya)** and **14% (Ghana)**. NkwaPay can then afford much more risk on each loan, because every good customer is worth much more.
4. **So the biggest open question isn't about defaults. It's about what a good customer is worth to NkwaPay over time.** The main numbers count one loan, as the brief's example does. The repeat-business version is fuller, but rests on a guess (half of declined customers leave for good).

**Decision for Step 5:**
- Keep the main cut-offs: **decline above 2.1% (Kenya) and 3.2% (Ghana)**.
- Report the range **1.9%–2.6% (Kenya) and 2.9%–4.1% (Ghana)** for the recovery and collection guesses.
- Report the repeat-business cut-offs (**9.6% Kenya, 14.2% Ghana**) as the most important alternative.
- **Germain:** when choosing the operating point (step 7), show the total cost at both the main and the repeat-business cut-offs, so the committee can see the trade-off.